# Crash Course — EHR 2차 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/ehr.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/ehr.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 1. 불멸 시간 편향을 눈으로 보기

효과가 0인 약입니다. 사망 시점은 약과 무관하게 정해지고, 약은 살아 있을 때만 시작할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 20000
death_day = rng.exponential(scale=80, size=n)
will_get = rng.random(n) < 0.4
start_day = rng.exponential(scale=5, size=n)
treated = will_get & (start_day < np.minimum(death_day, 28))
died28 = death_day < 28
print("잘못된 비교(한 번이라도 투약):", pd.Series(died28).groupby(treated).mean().round(3).to_dict())

alive3 = death_day >= 3
lm = pd.Series(died28[alive3]).groupby((will_get & (start_day < 3))[alive3]).mean().round(3)
print("landmark 3일:", lm.to_dict())

### 2. 교란 통제 전후: 단순 OR → 보정 OR, 그리고 성향점수 가중 후 균형

아픈 환자일수록 치료받고, 치료는 실제로 사망을 줄이도록(조건부 OR ≈ 0.67) 만든 가상 자료입니다.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(23)
n = 4000
age = rng.normal(65, 13, n)
sofa = np.clip(rng.poisson(5, n) + (age > 75), 0, 20)
lactate = np.exp(0.3 + 0.08 * sofa + rng.normal(0, 0.3, n))
treat = rng.random(n) < 1 / (1 + np.exp(-(-3.0 + 0.35 * sofa + 0.3 * np.log(lactate) + 0.01 * (age - 65))))
died = rng.random(n) < 1 / (1 + np.exp(-(-4.2 + 0.30 * sofa + 0.03 * (age - 65) + 0.5 * np.log(lactate) - 0.40 * treat)))
df = pd.DataFrame({"age": age, "sofa": sofa, "log_lactate": np.log(lactate),
                   "treat": treat.astype(int), "died": died.astype(int)})
covs = ["age", "sofa", "log_lactate"]

crude = np.exp(sm.Logit(df.died, sm.add_constant(df[["treat"]])).fit(disp=0).params["treat"])
adj = np.exp(sm.Logit(df.died, sm.add_constant(df[["treat"] + covs])).fit(disp=0).params["treat"])
print(f"단순 OR {crude:.2f} → 보정 OR {adj:.2f}")

ps = LogisticRegression(C=1e6, max_iter=1000).fit(df[covs], df.treat).predict_proba(df[covs])[:, 1]
w = np.where(df.treat == 1, df.treat.mean() / ps, (1 - df.treat.mean()) / (1 - ps))
for c in covs:
    t, u = df.treat == 1, df.treat == 0
    def s(wt):
        m1, m0 = np.average(df[c][t], weights=wt[t]), np.average(df[c][u], weights=wt[u])
        v1 = np.average((df[c][t] - m1) ** 2, weights=wt[t]); v0 = np.average((df[c][u] - m0) ** 2, weights=wt[u])
        return (m1 - m0) / np.sqrt((v1 + v0) / 2)
    print(f"{c:12s} SMD 전 {s(np.ones(n)):.2f} → IPW 후 {s(w):.2f}")